In [9]:
import json
import requests
import time

from datetime import datetime, timedelta, timezone


# FHIR API and Raw Lakehouse settings
FHIR_BASE_URL = "https://hapi.fhir.org/baseR4"

RESOURCE_TYPES = [
    "Patient",
    "Encounter",
    "Observation",
    "Condition"
]

PAGE_SIZE = 100
RAW_BASE_PATH = "Files/raw"
RAW_METADATA_PATH = "Files/raw_metadata"
REQUEST_TIMEOUT_SECONDS = 60

HEADERS = {
    "Accept": "application/fhir+json"
}

print("FHIR ingestion framework initialized.")
print(f"FHIR Base URL: {FHIR_BASE_URL}")
print(f"Page size: {PAGE_SIZE}")


StatementMeta(, 4e9d5086-6721-497c-bb31-6ff35a4fd612, 11, Finished, Available, Finished, False)

FHIR ingestion framework initialized.
FHIR Base URL: https://hapi.fhir.org/baseR4
Page size: 100


In [10]:
def get_fhir_response(
    url: str,
    params: dict | None = None,
    max_retries: int = 4,
    base_delay_seconds: int = 5
):
    # Retry transient API failures with exponential backoff.
    last_exception = None

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(
                url,
                params=params,
                headers=HEADERS,
                timeout=REQUEST_TIMEOUT_SECONDS
            )

            if response.status_code in [429, 500, 502, 503, 504]:
                print(
                    f"Transient HTTP {response.status_code}. "
                    f"Retry {attempt}/{max_retries}"
                )

                if attempt < max_retries:
                    delay = base_delay_seconds * (2 ** (attempt - 1))
                    print(f"Waiting {delay} seconds before retry...")
                    time.sleep(delay)
                    continue

            response.raise_for_status()
            return response

        except requests.exceptions.RequestException as exc:
            last_exception = exc

            print(
                f"Request failed on attempt "
                f"{attempt}/{max_retries}: "
                f"{type(exc).__name__}"
            )

            if attempt < max_retries:
                delay = base_delay_seconds * (2 ** (attempt - 1))
                print(f"Waiting {delay} seconds before retry...")
                time.sleep(delay)

    raise last_exception


StatementMeta(, 4e9d5086-6721-497c-bb31-6ff35a4fd612, 12, Finished, Available, Finished, False)

In [ ]:
# Pipeline parameter: number of previous UTC days to ingest.
DAYS_BACK = 3


In [11]:
# Build one incremental UTC window for each requested day.
def build_daily_windows(days_back: int = 3):
    today = datetime.now(timezone.utc).date()
    windows = []

    for i in range(days_back, 0, -1):
        window_date = today - timedelta(days=i)

        start_time = datetime(
            window_date.year,
            window_date.month,
            window_date.day,
            tzinfo=timezone.utc
        )

        end_time = start_time + timedelta(days=1)

        windows.append({
            "window_date": window_date.isoformat(),
            "start": start_time.isoformat().replace("+00:00", "Z"),
            "end": end_time.isoformat().replace("+00:00", "Z")
        })

    return windows


daily_windows = build_daily_windows(days_back=DAYS_BACK)

for window in daily_windows:
    print(
        f"{window['window_date']} | "
        f"{window['start']} -> {window['end']}"
    )


StatementMeta(, 4e9d5086-6721-497c-bb31-6ff35a4fd612, 13, Finished, Available, Finished, False)

2026-11-11 | 2026-11-11T00:00:00Z -> 2026-11-12T00:00:00Z
2026-11-12 | 2026-11-12T00:00:00Z -> 2026-11-13T00:00:00Z
2026-11-13 | 2026-11-13T00:00:00Z -> 2026-11-14T00:00:00Z


In [12]:
# Fetch all pages for one resource and daily _lastUpdated window.
def fetch_fhir_incremental_pages(
    resource_type: str,
    start_timestamp: str,
    end_timestamp: str
):
    params = {
        "_count": PAGE_SIZE,
        "_lastUpdated": [
            f"ge{start_timestamp}",
            f"lt{end_timestamp}"
        ]
    }

    url = f"{FHIR_BASE_URL}/{resource_type}"
    pages = []
    page_number = 1

    while url:
        extraction_timestamp = datetime.now(timezone.utc).isoformat()

        print(
            f"Fetching {resource_type} | "
            f"{start_timestamp} -> {end_timestamp} | "
            f"page {page_number}"
        )

        response = get_fhir_response(
            url=url,
            params=params if page_number == 1 else None
        )

        response_json = response.json()
        entries = response_json.get("entry", [])

        pages.append({
            "page_number": page_number,
            "request_url": response.url,
            "extraction_timestamp": extraction_timestamp,
            "response_json": response_json,
            "records_count": len(entries)
        })

        print(f"Records returned: {len(entries)}")

        next_url = None
        for link in response_json.get("link", []):
            if link.get("relation") == "next":
                next_url = link.get("url")
                break

        url = next_url
        page_number += 1

    print(f"Completed {resource_type}: {len(pages)} page(s)")
    return pages


StatementMeta(, 4e9d5086-6721-497c-bb31-6ff35a4fd612, 14, Finished, Available, Finished, False)

In [13]:
# Save the original Bundle without changing its JSON content.
def write_incremental_raw_page(
    resource_type: str,
    ingestion_date: str,
    page_number: int,
    response_json: dict
):
    resource_folder = resource_type.lower()

    folder_path = (
        f"{RAW_BASE_PATH}/"
        f"{resource_folder}/"
        f"ingestion_date={ingestion_date}"
    )

    notebookutils.fs.mkdirs(folder_path)

    file_path = (
        f"{folder_path}/"
        f"page_{page_number:03d}.json"
    )

    raw_content = json.dumps(
        response_json,
        ensure_ascii=False
    )

    notebookutils.fs.put(
        file_path,
        raw_content,
        True
    )

    return file_path


StatementMeta(, 4e9d5086-6721-497c-bb31-6ff35a4fd612, 15, Finished, Available, Finished, False)

In [14]:
# Replace the affected date partition so reruns stay idempotent.
def ingest_resource_for_window(
    resource_type: str,
    window: dict
):
    pages = fetch_fhir_incremental_pages(
        resource_type=resource_type,
        start_timestamp=window["start"],
        end_timestamp=window["end"]
    )

    ingestion_date = window["window_date"]
    resource_folder = resource_type.lower()

    raw_date_path = (
        f"{RAW_BASE_PATH}/"
        f"{resource_folder}/"
        f"ingestion_date={ingestion_date}"
    )

    # Remove the previous snapshot only after a successful API fetch.
    if notebookutils.fs.exists(raw_date_path):
        notebookutils.fs.rm(raw_date_path, True)

    metadata = []

    for page in pages:
        save_timestamp = datetime.now(timezone.utc).isoformat()

        file_path = write_incremental_raw_page(
            resource_type=resource_type,
            ingestion_date=ingestion_date,
            page_number=page["page_number"],
            response_json=page["response_json"]
        )

        metadata.append({
            "resource_type": resource_type,
            "window_date": ingestion_date,
            "page_number": page["page_number"],
            "api_url_or_params": page["request_url"],
            "extraction_timestamp": page["extraction_timestamp"],
            "data_saved_timestamp": save_timestamp,
            "records_extracted": page["records_count"],
            "raw_file_path": file_path,
            "status": "SUCCESS"
        })

        print(f"Saved: {file_path}")

    return metadata


StatementMeta(, 4e9d5086-6721-497c-bb31-6ff35a4fd612, 16, Finished, Available, Finished, False)

In [15]:
# Run the requested daily windows in API order.
all_metadata = []

for window in daily_windows:
    print("=" * 80)
    print(f"PROCESSING DATE: {window['window_date']}")
    print("=" * 80)

    for resource_type in RESOURCE_TYPES:
        print()
        print(f"STARTING RESOURCE: {resource_type}")

        resource_metadata = ingest_resource_for_window(
            resource_type=resource_type,
            window=window
        )

        all_metadata.extend(resource_metadata)

        print(f"COMPLETED RESOURCE: {resource_type}")

print()
print("=" * 80)
print("INGESTION COMPLETED")
print("=" * 80)

print(f"Total API pages: {len(all_metadata)}")

total_records = sum(
    item["records_extracted"]
    for item in all_metadata
)

print(f"Total records extracted: {total_records}")


StatementMeta(, 4e9d5086-6721-497c-bb31-6ff35a4fd612, 17, Finished, Available, Finished, False)

Ingestion Run ID: e976b12c-36df-4f51-bf78-ca6cca910fd7


## Save Ingestion Metadata
Keep API URL, extraction time, save time, and page counts outside the raw response.


In [ ]:
# Replace metadata for the selected dates to keep reruns idempotent.
for window in daily_windows:
    metadata_path = (
        f"{RAW_METADATA_PATH}/"
        f"ingestion_date={window['window_date']}"
    )

    if notebookutils.fs.exists(metadata_path):
        notebookutils.fs.rm(metadata_path, True)

metadata_by_date = {}

for item in all_metadata:
    metadata_by_date.setdefault(item["window_date"], []).append(item)

for ingestion_date, items in metadata_by_date.items():
    metadata_path = (
        f"{RAW_METADATA_PATH}/"
        f"ingestion_date={ingestion_date}"
    )

    notebookutils.fs.mkdirs(metadata_path)

    metadata_file = f"{metadata_path}/ingestion_metadata.json"
    metadata_json = "\n".join(
        json.dumps(item, ensure_ascii=False)
        for item in items
    )

    notebookutils.fs.put(metadata_file, metadata_json, True)
    print(f"Metadata saved: {metadata_file}")
